In [1]:
import openai
import os 
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import Client

In [2]:
qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

Download all data from qdrant

In [3]:
all_points = qdrant_client.scroll(
    collection_name="Amazon_Electronics_Products",
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

In [4]:
all_points

([Record(id=0, payload={'product_id': 'b86c3534-084b-449b-889e-50bd268ff964', 'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg', 'variant': 'MAIN', 'hi_res': None}], '

In [5]:
all_points[0][1].payload

{'product_id': '945a8ca0-282f-424a-83db-646304822e23',
 'text': 'Ce-H22B12-S1 4Kx2K Hdmi 4Port',
 'description': ['HDMI In - HDMI Out'],
 'images': [{'thumb': 'https://m.media-amazon.com/images/I/31OIMoOW70L._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/31OIMoOW70L._AC_.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/51qxU4Zd5TL._AC_SL1050_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41S98g84d0L._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41S98g84d0L._AC_.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/51HDwazbqNL._AC_SL1050_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41b68m0EuEL._AC_US40_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41b68m0EuEL._AC_.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/61LRQBRVGVL._AC_SL1050_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/31wG3YbResL._AC_US40_.jpg',
   'large': 'https:/

In [9]:
all_context = [
	{
		"id": data.id,
		"text": payload.get("text"),
        "product_id": payload.get("product_id"),
        "description": payload.get("description"),
        "images": payload.get("images"),
        "videos": payload.get("videos"),
        "features": payload.get("features"),	
        "price": payload.get("price"),
        "rating_number": payload.get("rating_number"),
        "main_category": payload.get("main_category"),
        "categories": payload.get("categories"),
        "store": payload.get("store"),
        "details": payload.get("details")
	}
	for data in all_points[0]
	if (payload := data.payload) is not None
]

In [11]:
all_context

[{'id': 0,
  'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
  'product_id': 'b86c3534-084b-449b-889e-50bd268ff964',
  'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'],
  'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
    'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg',
    'variant': 'MAIN',
    'hi_res': No

Render a prompt to generate synthetic Eval Reference Dataset

In [12]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "Suggested question",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context chunks.",
            },
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the retrieved chunks.",
            }
        }
    }
}

In [13]:
SYSTEM_PROMPT = """You are a helpful assistant for generating evaluation data for a retrieval-augmented generation (RAG) system.
Your task is to generate a set of questions that can be answered using the provided context chunks, which are reviews of products. For each question, you should also provide:
1. The IDs of the context chunks that could be used to answer the question.
2. An example answer grounded in the context chunks.
3. A reasoning explanation for why the question can be answered with the retrieved chunks.  
Make sure the questions are relevant to the content of the reviews and that the example answers are accurate and based on the information in the context chunks. The reasoning should clearly explain the connection between the question, the context chunks, and the answer.
4. Construct 5 questions with multiple choice answers, where each question has 4 options (A, B, C, D) and only one correct answer. The options should be plausible and based on the context chunks, but only one should be correct.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>
I need to be able to parse the json output
"""

USER_PROMPT = f"""Here is the lost of chunks, each list element is a dictionary with id and text:
{all_context}
Please generate the evaluation data as per the system prompt instructions.
"""

In [14]:
print(SYSTEM_PROMPT)

You are a helpful assistant for generating evaluation data for a retrieval-augmented generation (RAG) system.
Your task is to generate a set of questions that can be answered using the provided context chunks, which are reviews of products. For each question, you should also provide:
1. The IDs of the context chunks that could be used to answer the question.
2. An example answer grounded in the context chunks.
3. A reasoning explanation for why the question can be answered with the retrieved chunks.  
Make sure the questions are relevant to the content of the reviews and that the example answers are accurate and based on the information in the context chunks. The reasoning should clearly explain the connection between the question, the context chunks, and the answer.
4. Construct 5 questions with multiple choice answers, where each question has 4 options (A, B, C, D) and only one correct answer. The options should be plausible and based on the context chunks, but only one should be cor

In [15]:
print(USER_PROMPT)

Here is the lost of chunks, each list element is a dictionary with id and text:
[{'id': 0, 'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'product_id': 'b86c3534-084b-449b-889e-50bd268ff964', 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/ima

In [16]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Retrieve API keys from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')
qdrant_url = os.getenv('QDRANT_URL')
qdrant_api_key = os.getenv('QDRANT_API_KEY')

# Verify keys are loaded
print(f"OpenAI API Key present: {bool(openai_api_key)}")
print(f"Google API Key present: {bool(google_api_key)}")
print(f"Qdrant URL present: {bool(qdrant_url)}")
print(f"Qdrant API Key present: {bool(qdrant_api_key)}")

OpenAI API Key present: True
Google API Key present: False
Qdrant URL present: True
Qdrant API Key present: False


In [17]:
response  = openai.chat.completions.create(
    model="gpt-5-mini",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort = 'minimal'
)

print(response.choices[0].message.content)

{
  "questions": [
    {
      "id": "Q1",
      "question": "Which product includes a built-in fingerprint biometric authentication and AES 128-bit hardware encryption?",
      "context_ids": [
        95
      ],
      "answer": "LaCie Rugged SAFE All-Terrain (id 95) — it features biometric authentication (fingerprint) for up to ten users and AES 128-bit hardware encryption.",
      "reasoning": "Chunk 95 (LaCie Rugged All-Terrain Safe 500 GB) describes biometric authentication for up to ten users and AES 128-bit hardware encryption in its features and manufacturer description, which directly answers the question."
    },
    {
      "id": "Q2",
      "question": "Which product explicitly states compatibility with MacBook Pro 13-inch model numbers A2338, A2289, and A2251?",
      "context_ids": [
        2
      ],
      "answer": "Digi-Tatoo Decal Skin for MacBook Pro 13 inch (id 2) — its features warn to identify the model number and state it only fits models A2338, A2289, and A225

In [18]:
import json 
json_output = response.choices[0].message.content
json_output = json.loads(json_output)

In [19]:
print("Type of json_output:", type(json_output))
print("Length or keys:", len(json_output) if isinstance(json_output, (list, dict)) else "N/A")

if isinstance(json_output, dict):
    print("Dict keys:", list(json_output.keys()))
elif isinstance(json_output, list) and len(json_output) > 0:
    print("List of", len(json_output), "items; first item keys:", 
          list(json_output[0].keys()) if isinstance(json_output[0], dict) else "not a dict")

Type of json_output: <class 'dict'>
Length or keys: 2
Dict keys: ['questions', 'multiple_choice']


In [20]:
# EXTRACT questions from various possible structures
if isinstance(json_output, dict):
    # Try common keys for questions array
    questions = (json_output.get("questions") or 
                 json_output.get("multiple_choice_questions") or 
                 json_output.get("data") or [])
elif isinstance(json_output, list):
    questions = json_output
else:
    questions = []

print(f"Extracted {len(questions)} questions")

# NORMALIZE and FIX each question structure
normalized_questions = []
for item in questions:
    if not isinstance(item, dict):
        continue
    
    # Map various possible key names to standard ones
    normalized = {
        "question": item.get("question") or item.get("q") or item.get("query") or "",
        "chunk_ids": item.get("chunk_ids") or item.get("relevant_chunks") or item.get("chunk_ids") or [],
        "answer_example": item.get("answer_example") or item.get("answer") or item.get("expected_answer") or ""
    }
    
    # Only keep non-empty questions
    if normalized["question"]:
        normalized_questions.append(normalized)

print(f"Normalized {len(normalized_questions)} questions with content")
if normalized_questions:
    print("First normalized question:", normalized_questions[0])

Extracted 10 questions
Normalized 10 questions with content
First normalized question: {'question': 'Which product includes a built-in fingerprint biometric authentication and AES 128-bit hardware encryption?', 'chunk_ids': [], 'answer_example': 'LaCie Rugged SAFE All-Terrain (id 95) — it features biometric authentication (fingerprint) for up to ten users and AES 128-bit hardware encryption.'}


In [21]:
# DIAGNOSTIC: Inspect json_output and first question structure
print("json_output type:", type(json_output))
print("json_output length:", len(json_output) if isinstance(json_output, (list, dict)) else "N/A")

if isinstance(json_output, dict):
    print("Dict keys:", list(json_output.keys()))
    first_q = next(iter(json_output.values())) if json_output else None
elif isinstance(json_output, list) and json_output:
    first_q = json_output[0]
else:
    first_q = None

if first_q and isinstance(first_q, dict):
    print("\nFirst question/item keys:", list(first_q.keys()))
    print("Full structure:", first_q)
else:
    print("Could not extract first item or not a dict")

json_output type: <class 'dict'>
json_output length: 2
Dict keys: ['questions', 'multiple_choice']
Could not extract first item or not a dict


In [22]:
if isinstance(json_output, dict):
    questions = (json_output.get("questions") or 
                 json_output.get("multiple_choice_questions") or 
                 json_output.get("data") or [])
elif isinstance(json_output, list):
    questions = json_output
else:
    questions = []

print(f"Extracted {len(questions)} questions")

# Normalize and inspect
normalized_questions = []
for idx, item in enumerate(questions):
    if not isinstance(item, dict):
        print(f"  Item {idx} is not a dict, skipping")
        continue
    
    # Show keys for first few items
    if idx < 3:
        print(f"\nItem {idx} keys: {list(item.keys())}")
    
    # Try multiple key variations for answer
    answer = (item.get("answer_example") or 
              item.get("answer") or 
              item.get("expected_answer") or 
              item.get("suggested_answer") or "")
    
    # Try multiple key variations for chunk_ids  
    chunk_ids = (item.get("chunk_ids") or 
                 item.get("relevant_chunks") or 
                 item.get("related_chunk_ids") or 
                 item.get("source_chunk_ids") or [])
    
    question = (item.get("question") or 
                item.get("q") or 
                item.get("query") or "")
    
    normalized = {
        "question": question.strip() if isinstance(question, str) else "",
        "chunk_ids": chunk_ids if isinstance(chunk_ids, list) else [],
        "answer_example": answer.strip() if isinstance(answer, str) else ""
    }
    
    if normalized["question"]:  # Only keep non-empty questions
        normalized_questions.append(normalized)

print(f"\nNormalized {len(normalized_questions)} questions")
if normalized_questions:
    print("First normalized:", normalized_questions[0])

Extracted 10 questions

Item 0 keys: ['id', 'question', 'context_ids', 'answer', 'reasoning']

Item 1 keys: ['id', 'question', 'context_ids', 'answer', 'reasoning']

Item 2 keys: ['id', 'question', 'context_ids', 'answer', 'reasoning']

Normalized 10 questions
First normalized: {'question': 'Which product includes a built-in fingerprint biometric authentication and AES 128-bit hardware encryption?', 'chunk_ids': [], 'answer_example': 'LaCie Rugged SAFE All-Terrain (id 95) — it features biometric authentication (fingerprint) for up to ten users and AES 128-bit hardware encryption.'}


In [23]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )
    return response.data[0].embedding

In [24]:
json_output

{'questions': [{'id': 'Q1',
   'question': 'Which product includes a built-in fingerprint biometric authentication and AES 128-bit hardware encryption?',
   'context_ids': [95],
   'answer': 'LaCie Rugged SAFE All-Terrain (id 95) — it features biometric authentication (fingerprint) for up to ten users and AES 128-bit hardware encryption.',
   'reasoning': 'Chunk 95 (LaCie Rugged All-Terrain Safe 500 GB) describes biometric authentication for up to ten users and AES 128-bit hardware encryption in its features and manufacturer description, which directly answers the question.'},
  {'id': 'Q2',
   'question': 'Which product explicitly states compatibility with MacBook Pro 13-inch model numbers A2338, A2289, and A2251?',
   'context_ids': [2],
   'answer': 'Digi-Tatoo Decal Skin for MacBook Pro 13 inch (id 2) — its features warn to identify the model number and state it only fits models A2338, A2289, and A2251.',
   'reasoning': 'Chunk 2 (Digi-Tatoo MacBook skin) includes a WARNING feature

In [25]:
# Safely inspect the first generated question (avoid KeyError if json_output is a dict)
if isinstance(json_output, dict):
    questions = json_output.get("questions") or json_output.get("multiple_choice_questions") or []
    if questions:
        questions[0]
    else:
        json_output
else:
    json_output

In [26]:
len(json_output)

2

In [27]:
points = qdrant_client.scroll(collection_name="Amazon_Electronics_Products", scroll_filter=Filter(must=[FieldCondition(key="product_id", match=MatchValue(value="b86c3534-084b-449b-889e-50bd268ff964"))]), limit=100, with_payload=True)[0]

In [28]:
points

[Record(id=0, payload={'product_id': 'b86c3534-084b-449b-889e-50bd268ff964', 'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg', 'variant': 'MAIN', 'hi_res': None}], 'v

In [30]:
all_points = qdrant_client.scroll(collection_name="Amazon_Electronics_Products", limit=5, with_payload=True)[0]
for p in all_points: print(p.id, p.payload)

0 {'product_id': 'b86c3534-084b-449b-889e-50bd268ff964', 'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg', 'variant': 'MAIN', 'hi_res': None}], 'videos': [], 'price':

In [31]:
points, next_offset = qdrant_client.scroll(
    collection_name="Amazon_Electronics_Products",
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="product_id",
                match=MatchValue(value="b86c3534-084b-449b-889e-50bd268ff964")
            )
        ]
    ),
    limit=100,
    offset=None,
    with_payload=True,
    with_vectors=False
)

# all payloads
all_payloads = [point.payload for point in points]
all_payloads

[{'product_id': 'b86c3534-084b-449b-889e-50bd268ff964',
  'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
  'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'],
  'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
    'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg',
    'variant': 'MAIN',
    'hi_res': None}],
  'vi

In [32]:
for point in points:
    print("ID:", point.id)
    print("Payload:", point.payload)
    print("-" * 50)

ID: 0
Payload: {'product_id': 'b86c3534-084b-449b-889e-50bd268ff964', 'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg', 'variant': 'MAIN', 'hi_res': None}], 'videos':

In [33]:
import pandas as pd

df_results = pd.json_normalize([point.payload for point in points])
df_results

,product_id,text,description,images,videos,price,rating_number,main_category,categories,store,features,details.Date First Available,details.Manufacturer
0,b86c3534-084b-449b-889e-50bd268ff964,FS-1051 FATSHARK TELEPORTER V3 HEADSET,[Teleporter V3 The “Teleporter V3” kit sets a ...,[{'thumb': 'https://m.media-amazon.com/images/...,[],96.551146,6,All Electronics,"[Electronics, Television & Video, Video Glasses]",Fat Shark,[],"August 2, 2014",Fatshark


In [34]:
texts = [point.payload.get("text") for point in points]
texts

['FS-1051 FATSHARK TELEPORTER V3 HEADSET']

In [35]:
points

[Record(id=0, payload={'product_id': 'b86c3534-084b-449b-889e-50bd268ff964', 'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg', 'variant': 'MAIN', 'hi_res': None}], 'v

In [36]:
def get_details_for_chunk_ids(chunk_ids):
    details = []
    for chunk_id in chunk_ids:
        point = qdrant_client.scroll(
            collection_name="Amazon_Electronics_Products",
            scroll_filter=Filter(
                must=[
                    FieldCondition(
                        key="product_id",
                        match=MatchValue(value=chunk_id)
                    )
                ]
            ),
            limit=1,
            offset=None,
            with_payload=True,
            with_vectors=False
        )[0]
        if point:
            details.append(point[0].payload)
    return details

In [37]:
get_details_for_chunk_ids(['b86c3534-084b-449b-889e-50bd268ff964', 'c67b6305-54b3-4a14-816a-532c3d2a108c'])

[{'product_id': 'b86c3534-084b-449b-889e-50bd268ff964',
  'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
  'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'],
  'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg',
    'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg',
    'variant': 'MAIN',
    'hi_res': None}],
  'vi

Create eval dataset in langsmith

In [38]:
client = Client(api_key=os.getenv("LANGSMITH_API_KEY"))

In [39]:
dataset_name = "rag-evaluation-dataset"
dataset = client.create_dataset(dataset_name=dataset_name, description="Dataset for evaluating RAG system performance on question answering tasks based on product reviews.")

In [40]:
dataset

Dataset(name='rag-evaluation-dataset', description='Dataset for evaluating RAG system performance on question answering tasks based on product reviews.', data_type=<DataType.kv: 'kv'>, id=UUID('64bf9bd5-c777-4426-a769-577f772001cf'), created_at=datetime.datetime(2026, 5, 21, 5, 14, 13, 360607, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 5, 21, 5, 14, 13, 360607, tzinfo=TzInfo(0)), example_count=None, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'sdk_version': '0.8.4', 'library': 'langsmith', 'platform': 'macOS-26.5-arm64-arm-64bit', 'runtime': 'python', 'py_implementation': 'CPython', 'runtime_version': '3.12.5', 'langchain_version': '1.3.0', 'langchain_core_version': '1.4.0'}})

In [36]:
import json

# Save embeddings to a separate file or dict
embeddings_map = {}
for chunk in all_context:
    embeddings_map[chunk["id"]] = get_embedding(chunk["text"])

# Store embeddings (optional: save to file)
with open("chunk_embeddings.json", "w") as f:
    json.dump(embeddings_map, f)

# Create examples without storing embeddings in LangSmith
# (Keep references to chunk IDs in metadata instead)
for item in questions:
    if not isinstance(item, dict):
        continue
    
    question = item.get("question")
    chunk_ids = item.get("chunk_ids", [])
    
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": question},
        outputs={"answer": item.get("answer_example", "")},
        metadata={
            "chunk_ids": chunk_ids,
            "embedding_model": "text-embedding-3-small"
        }
    )

In [39]:
embeddings_map.keys()

dict_keys([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

In [40]:
embeddings_map

{0: [-0.0206756591796875,
  -0.04010009765625,
  0.0303192138671875,
  -0.0390625,
  -0.0233917236328125,
  -0.040863037109375,
  0.0281219482421875,
  0.03973388671875,
  0.0386962890625,
  -0.033782958984375,
  0.0204010009765625,
  0.007526397705078125,
  -0.012939453125,
  0.025115966796875,
  -0.01149749755859375,
  -0.055206298828125,
  -0.0021152496337890625,
  -0.0194244384765625,
  -0.0203857421875,
  0.0135650634765625,
  0.037017822265625,
  0.0232391357421875,
  -0.00589752197265625,
  0.00713348388671875,
  -0.01401519775390625,
  -0.051116943359375,
  0.03125,
  -0.0153045654296875,
  0.024444580078125,
  0.0099334716796875,
  0.012908935546875,
  -0.038177490234375,
  0.035858154296875,
  -0.0132904052734375,
  -0.042633056640625,
  -0.002452850341796875,
  -0.0005397796630859375,
  0.00478363037109375,
  -0.0291900634765625,
  -0.01226806640625,
  0.04412841796875,
  0.0018873214721679688,
  0.0167083740234375,
  0.04913330078125,
  0.017974853515625,
  -0.0231018066406

In [44]:
# POPULATE: Build all_context_with_embeddings (add embeddings to each chunk)
all_context_with_embeddings = []
for chunk in all_context:
    chunk_embedding = get_embedding(chunk["text"])
    chunk_copy = chunk.copy()
    chunk_copy["embedding"] = chunk_embedding
    all_context_with_embeddings.append(chunk_copy)

print(f"Populated all_context_with_embeddings with {len(all_context_with_embeddings)} chunks")

# INSERT INTO LANGSMITH: Use populated embeddings and fixed questions
chunk_id_to_embedding = {}
for chunk in all_context_with_embeddings:
    chunk_id = str(chunk.get("id", ""))
    if chunk_id:
        chunk_id_to_embedding[chunk_id] = chunk.get("embedding", [])

print(f"Built mapping for {len(chunk_id_to_embedding)} chunks")

# Loop through normalized questions and insert
inserted = 0
skipped = 0

for item in normalized_questions:
    question = item.get("question", "").strip()
    answer = item.get("answer_example", "").strip()
    chunk_ids = item.get("chunk_ids", [])
    
    # Skip if missing critical fields
    if not question:
        skipped += 1
        continue
    
    # Look up embeddings for referenced chunks
    chunk_embeddings = []
    for cid in chunk_ids:
        cid_str = str(cid)
        if cid_str in chunk_id_to_embedding:
            chunk_embeddings.append(chunk_id_to_embedding[cid_str])
    
    # Create example (answer can be empty, but question must exist)
    client.create_example(
        dataset_id=dataset.id,
        inputs={"question": question},
        outputs={
            "answer": answer if answer else question  # Use question as fallback
        },
        metadata={
            "chunk_ids": [str(cid) for cid in chunk_ids],
            "embedding_model": "text-embedding-3-small",
            "has_answer": bool(answer)
        }
    )
    inserted += 1

print(f"\nInserted {inserted} examples into dataset (skipped {skipped} with no question)")

Populated all_context_with_embeddings with 100 chunks
Built mapping for 100 chunks

Inserted 10 examples into dataset (skipped 0 with no question)


In [42]:
print("all_context length:", len(all_context))
if all_context:
    print("First chunk in all_context:", all_context[0])
else:
    print("WARNING: all_context is empty!")

all_context length: 100
First chunk in all_context: {'id': 0, 'text': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET', 'product_id': 'b86c3534-084b-449b-889e-50bd268ff964', 'description': ['Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV headset with integrated NexwaveRF receiver technology while simultaneously recording onboard HD footage with the included “PilotHD” camera. The “Teleporter V3” kit comes complete with everything you need to step into the cockpit of your FPV vehicle. We’ve included our powerful 250mW 5.8Ghz transmitter, 25 degree FOV headset (largest QVGA display available), the brand new “PilotHD” camera with live AV out and all the cables, antennas and connectors needed.'], 'images': [{'thumb': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_US40_.jpg', 'large': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg', 

In [43]:
# POPULATE: Build all_context_with_embeddings (add embeddings to each chunk)
all_context_with_embeddings = []
for chunk in all_context:
    chunk_embedding = get_embedding(chunk["text"])
    chunk_copy = chunk.copy()
    chunk_copy["embedding"] = chunk_embedding
    all_context_with_embeddings.append(chunk_copy)

print(f"Populated all_context_with_embeddings with {len(all_context_with_embeddings)} chunks")
if all_context_with_embeddings:
    print("First chunk (keys):", list(all_context_with_embeddings[0].keys()))
    print("First chunk ID:", all_context_with_embeddings[0].get("id"))

Populated all_context_with_embeddings with 100 chunks
First chunk (keys): ['id', 'text', 'product_id', 'description', 'images', 'videos', 'features', 'price', 'rating_number', 'main_category', 'categories', 'store', 'details', 'embedding']
First chunk ID: 0
